# Phase 6: Item-Based Collaborative Filtering Engine
## Behavioral Recommendation Modeling, Sparse Matrix Similarity & Temporal Evaluation

---

### 1. Title & Objective
This notebook implements an **Item-Based Collaborative Filtering (CF)** recommendation engine developed in Phase 6. While Phases 3–4 built a content-based recommendation system from movie metadata, Phase 6 incorporates **actual user behavioral interaction data** using the **MovieLens latest-small** dataset.

**Key Objectives**:
1. Load and validate genuine user-item rating interaction data.
2. Construct a sparse Item $\times$ User rating matrix and compute sparse Item-Item Cosine Similarity.
3. Generate personalized collaborative recommendations based on user preference aggregation.
4. Perform a **leakage-safe temporal evaluation** using Phase 5 evaluator metrics (Precision@K, Recall@K, NDCG@K).

### 2. Dataset Source & MovieLens Metadata Overview
> [!IMPORTANT]
> **Dataset Policy & Disclaimer**:
> The TMDB 5000 dataset contains rich movie metadata (genres, keywords, cast, crew, overviews) but **does NOT contain multi-user rating histories or interaction logs**.
> 
> Therefore, Phase 6 uses the **MovieLens latest-small** dataset created by GroupLens Research at the University of Minnesota. It contains 100,836 genuine ratings across 610 users and 9,724 movies on a 5-star scale (0.5 to 5.0 in 0.5 increments).
> 
> **DO NOT** assume MovieLens `movieId` matches TMDB `id`. MovieLens IDs are used internally for collaborative filtering.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure root path resolution for src module
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.collaborative_filter import (
    ItemBasedCollaborativeRecommender,
    temporal_train_test_split,
    evaluate_collaborative_model,
)
from src.evaluator import precision_at_k, recall_at_k, ndcg_at_k

print("Modules imported successfully.")

### 3. Load MovieLens Data
We load `ratings.csv` and `movies.csv` from `data/raw/movielens/ml-latest-small/`.

In [ ]:
project_root = Path(os.getcwd()).parent
ml_dir = project_root / "data" / "raw" / "movielens" / "ml-latest-small"
ratings_path = ml_dir / "ratings.csv"
movies_path = ml_dir / "movies.csv"

if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        f"MovieLens dataset missing from {ml_dir}. Please run scratch/download_movielens.py to download it."
    )

ratings_df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path)

print(f"Loaded ratings.csv: {len(ratings_df):,} rows")
print(f"Loaded movies.csv: {len(movies_df):,} rows")
print(f"Unique Users: {ratings_df['userId'].nunique()}")
print(f"Unique Movies Rated: {ratings_df['movieId'].nunique()}")
print(f"Rating Scale: {ratings_df['rating'].min()} to {ratings_df['rating'].max()}")

print("\nSample Ratings:")
print(ratings_df.head().to_string(index=False))

print("\nRating Distribution:")
print(ratings_df["rating"].value_counts().sort_index().to_dict())

### 4. Data Validation & Preprocessing Statistics
We instantiate `ItemBasedCollaborativeRecommender` and validate structure, nulls, numeric ranges, duplicate ratings (latest timestamp policy), and minimum rating thresholds per movie and per user.

In [ ]:
cf_recommender = ItemBasedCollaborativeRecommender(
    min_ratings_per_movie=5,
    min_ratings_per_user=5,
    movies_df=movies_df,
)
cf_recommender.fit(ratings_df)

stats = cf_recommender.stats
print("=" * 60)
print("PREPROCESSING & FILTERING STATISTICS")
print("=" * 60)
print(f"Raw Users            : {stats['raw_users']:,}")
print(f"Raw Movies           : {stats['raw_movies']:,}")
print(f"Raw Ratings          : {stats['raw_ratings']:,}")
print(f"Filtered Users       : {stats['filtered_users']:,}")
print(f"Filtered Movies      : {stats['filtered_movies']:,}")
print(f"Filtered Ratings     : {stats['filtered_ratings']:,}")
print(f"Matrix Sparsity      : {stats['sparsity'] * 100:.2f}%")

### 5. Collaborative Filtering Architecture Concept
```
User Ratings  ──>  Item x User Matrix R  ──>  Cosine Similarity S(i, j)  ──>  Score Accumulation  ──>  Exclusion & Top-K
```

**Mathematical Formulation**:
1. **Item-Item Cosine Similarity**:
   $$S(i, j) = \frac{\mathbf{R}_i \cdot \mathbf{R}_j}{\|\mathbf{R}_i\| \|\mathbf{R}_j\|}$$
   with diagonal $S(i, i) = 0.0$ to exclude self-recommendation.

2. **Personalized Candidate Scoring**:
   $$\text{score}(c) = \sum_{m \in \text{user\_pos\_movies}} S(m, c) \times r(u, m)$$
   where candidate movie $c$ is strictly unrated by user $u$.

### 6. Build Item-Based Model
The fitted `ItemBasedCollaborativeRecommender` holds the sparse Item $\times$ User matrix $R$ and Cosine Similarity matrix $S$.

In [ ]:
print(f"Fitted CSR Matrix Shape : {cf_recommender.item_user_matrix.shape}")
print(f"Similarity Matrix Shape : {cf_recommender.similarity_matrix.shape}")
print(f"Is Self-Similarity Diagonal Zeroed: {np.all(np.diag(cf_recommender.similarity_matrix) == 0.0)}")

### 7. Similar Movie Example
We query item neighbors for *Toy Story (1995)* (MovieLens `movieId=1`).

In [ ]:
query_movie_id = 1  # Toy Story (1995)
similar_items = cf_recommender.get_similar_items(query_movie_id, top_n=5)

query_title = cf_recommender.title_map.get(query_movie_id, f"Movie_{query_movie_id}")
print(f"Top 5 Most Similar Movies to '{query_title}' (movieId={query_movie_id}):")
df_sims = pd.DataFrame(similar_items)
print(df_sims.to_string(index=False))

### 8. Personalized Recommendation Example
We generate recommendations for target User ID `1`.

In [ ]:
target_user_id = 1
user1_recs = cf_recommender.recommend(target_user_id, top_n=5, positive_threshold=4.0)

user1_pos_history = [
    (cf_recommender.title_map.get(m_id, f"Movie_{m_id}"), r)
    for m_id, r in cf_recommender.user_history[target_user_id].items()
    if r >= 4.0
][:5]

print(f"User {target_user_id} Sample Positive Ratings (>=4.0): {user1_pos_history}")
print(f"\nTop 5 Personalized Collaborative Recommendations for User {target_user_id}:")
df_user_recs = pd.DataFrame(user1_recs)
print(df_user_recs.to_string(index=False))

### 9. Leakage-Safe Temporal Evaluation Protocol
To evaluate collaborative recommendations without test data leakage:
1. **Chronological Split**: Ratings per user are sorted chronologically by timestamp. Earliest 80% are training history; latest 20% positive ratings ($\ge 4.0$) are held-out test targets.
2. **Model Fitting Constraint**: The collaborative model is fitted **STRICTLY on the training ratings split**. Held-out test ratings are NEVER seen during model fitting.
3. **Data Leakage Safeguards**: Programmatically verify that test targets are absent from training history and that already-rated training movies are never recommended.

In [ ]:
train_df, test_scenarios = temporal_train_test_split(
    ratings_df, test_ratio=0.2, min_user_ratings=10, positive_threshold=4.0
)

# Verify data leakage safeguards for first 5 test scenarios
for sc in test_scenarios[:5]:
    hist_set = set(sc["train_history_ids"])
    held_out_set = set(sc["held_out_relevant_ids"])
    overlap = hist_set & held_out_set
    assert len(overlap) == 0, f"Data leakage detected for user {sc['userId']}: {overlap}"

print(f"Created training split with {len(train_df):,} ratings across {train_df['userId'].nunique()} users.")
print(f"Created {len(test_scenarios):,} test scenarios. Data leakage verification PASSED.")

### 10. Metric Evaluation (K = 3, 5, 10)
We evaluate the model across $K \in \{3, 5, 10\}$ using Phase 5 evaluator functions.

In [ ]:
eval_results = evaluate_collaborative_model(
    ratings_df,
    k_values=[3, 5, 10],
    min_ratings_per_movie=5,
    min_ratings_per_user=5,
    min_user_ratings=10,
    test_ratio=0.2,
    positive_threshold=4.0,
    max_eval_users=50,
    movies_df=movies_df,
)

print("Evaluation completed successfully.")

### 11. Aggregate Results Summary Table
We format the evaluation metrics across $K = 3, 5, 10$ into a pandas DataFrame.

In [ ]:
rows = []
for k_val, res in eval_results["eval_by_k"].items():
    rows.append({
        "Cutoff Rank (K)": f"K = {k_val}",
        "Evaluated Users": res["num_users"],
        "Held-Out Target Items": res["total_held_out_items"],
        "Mean Precision@K": res["mean_precision"],
        "Mean Recall@K": res["mean_recall"],
        "Mean NDCG@K": res["mean_ndcg"],
    })

df_summary = pd.DataFrame(rows)
print("=" * 75)
print("COLLABORATIVE FILTERING TEMPORAL EVALUATION SUMMARY")
print("=" * 75)
print(df_summary.to_string(index=False))

### 12. Interpretation & Limitations
1. **Behavioral Recommendation Strength**: Item-Based Collaborative Filtering captures genuine co-rating user patterns that content metadata alone cannot discover.
2. **Leakage Safeguards**: Model fitting on training data before testing guarantees zero data leakage.
3. **Limitations**:
   - **Cold-Start**: New items or users without prior interaction history cannot receive collaborative recommendations.
   - **Sparsity**: Sparse matrices ($> 98\%$ sparse) limit candidate overlap.
   - **Popularity Bias**: Highly rated popular items accumulate more similarity weight.
   - **Offline Disclaimer**: Offline metrics on MovieLens do **NOT** guarantee real-world production performance.